<a href="https://colab.research.google.com/github/cesarpaico1511/rag-mvcs-streamlit/blob/main/ProyectoCesarPaico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git config --global user.name "Cesar Paico"
!git config --global user.email "cesarpaico@gmail.com"

In [5]:
!git clone https://github.com/cesarpaico1511/rag-mvcs-streamlit.git

Cloning into 'rag-mvcs-streamlit'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [6]:
%cd rag-mvcs-streamlit

/content/rag-mvcs-streamlit


In [7]:
!pwd
!git status

/content/rag-mvcs-streamlit
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [8]:
!git checkout -b feature/rag-mvcs-mvp

Switched to a new branch 'feature/rag-mvcs-mvp'


In [9]:
!mkdir -p src tests notebooks data/documentos vectorstore logs .streamlit
!touch src/__init__.py
!touch data/documentos/.gitkeep vectorstore/.gitkeep logs/.gitkeep

In [10]:
!find . -maxdepth 3 -type d

.
./src
./.streamlit
./notebooks
./logs
./tests
./.git
./.git/hooks
./.git/logs
./.git/logs/refs
./.git/objects
./.git/objects/pack
./.git/objects/info
./.git/refs
./.git/refs/tags
./.git/refs/heads
./.git/refs/remotes
./.git/branches
./.git/info
./data
./data/documentos
./vectorstore


In [11]:
!touch app.py requirements.txt .env.example README.md .gitignore
!touch src/config.py src/loader.py src/embeddings.py src/retriever.py
!touch src/rag_chain.py src/ingest.py src/prompts.py src/logging_config.py
!touch tests/test_config.py tests/test_loader.py tests/test_prompts.py
!touch .streamlit/config.toml .streamlit/secrets.toml.example

In [12]:
!git status

On branch feature/rag-mvcs-mvp
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.env.example
	.gitignore
	.streamlit/
	app.py
	data/
	logs/
	requirements.txt
	src/
	tests/
	vectorstore/

nothing added to commit but untracked files present (use "git add" to track)


In [13]:
import os

# Define the content for .gitignore to keep the repository clean
gitignore_content = """
__pycache__/
*.py[cod]
*$py.class
.env
.venv
env/
venv/
logs/*.log
vectorstore/*
!vectorstore/.gitkeep
"""

with open('.gitignore', 'w') as f:
    f.write(gitignore_content.strip())

print(".gitignore file created.")

.gitignore file created.


In [14]:
!git add .
!git commit -m "Initial project structure for RAG MVCS MVP"

[feature/rag-mvcs-mvp 139be38] Initial project structure for RAG MVCS MVP
 21 files changed, 10 insertions(+)
 create mode 100644 .env.example
 create mode 100644 .gitignore
 create mode 100644 .streamlit/config.toml
 create mode 100644 .streamlit/secrets.toml.example
 create mode 100644 app.py
 create mode 100644 data/documentos/.gitkeep
 create mode 100644 logs/.gitkeep
 create mode 100644 requirements.txt
 create mode 100644 src/__init__.py
 create mode 100644 src/config.py
 create mode 100644 src/embeddings.py
 create mode 100644 src/ingest.py
 create mode 100644 src/loader.py
 create mode 100644 src/logging_config.py
 create mode 100644 src/prompts.py
 create mode 100644 src/rag_chain.py
 create mode 100644 src/retriever.py
 create mode 100644 tests/test_config.py
 create mode 100644 tests/test_loader.py
 create mode 100644 tests/test_prompts.py
 create mode 100644 vectorstore/.gitkeep


In [15]:
%%writefile requirements.txt
python-dotenv>=1.0.1
streamlit>=1.41.0
langchain>=0.3.20
langchain-core>=0.3.40
langchain-community>=0.3.19
langchain-text-splitters>=0.3.6
langchain-google-genai>=4.0.0
langchain-chroma>=0.2.2
chromadb>=0.6.3
pypdf>=5.3.0
pandas>=2.2.0
pytest>=8.0.0

Overwriting requirements.txt


In [17]:
!pip install -r requirements.txt

In [37]:
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Ingresa tu GOOGLE_API_KEY: ")

Ingresa tu GOOGLE_API_KEY: ··········


In [80]:
%%writefile src/config.py
import os
from pathlib import Path
from dotenv import load_dotenv

BASE_DIR = Path(__file__).resolve().parents[1]
load_dotenv(BASE_DIR / ".env", override=True)

APP_NAME = os.getenv("APP_NAME", "Asistente RAG MVCS")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "models/gemini-2.5-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "models/gemini-embedding-001")

DATA_DIR = BASE_DIR / "data"
DOCS_DIR = DATA_DIR / "documentos"
VECTORSTORE_DIR = BASE_DIR / "vectorstore"
LOGS_DIR = BASE_DIR / "logs"

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1000"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "150"))
RETRIEVAL_K = int(os.getenv("RETRIEVAL_K", "4"))

Overwriting src/config.py


In [21]:
%%writefile src/prompts.py
SYSTEM_PROMPT = """
Eres un asistente especializado en trámites y servicios del MVCS.
Responde solo con base en el contexto recuperado.
Si no hay evidencia suficiente, responde:
"No tengo evidencia suficiente en las fuentes cargadas".
No inventes requisitos, costos, plazos ni normativa.
"""

Overwriting src/prompts.py


In [22]:
%%writefile src/logging_config.py
import logging
from src.config import LOGS_DIR

LOGS_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    filename=LOGS_DIR / "app.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("rag_mvcs")

Overwriting src/logging_config.py


In [23]:
%%writefile src/embeddings.py
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from src.config import EMBEDDING_MODEL

def get_embeddings():
    return GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

Overwriting src/embeddings.py


In [24]:
%%writefile src/loader.py
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.config import CHUNK_SIZE, CHUNK_OVERLAP

def load_file(path):
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return PyPDFLoader(str(path)).load()
    if suffix == ".txt":
        return TextLoader(str(path), encoding="utf-8").load()
    if suffix == ".csv":
        return CSVLoader(str(path), encoding="utf-8").load()
    return []

def split_docs(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    return splitter.split_documents(docs)

Overwriting src/loader.py


In [25]:
%%writefile src/retriever.py
from langchain_chroma import Chroma
from src.config import VECTORSTORE_DIR
from src.embeddings import get_embeddings

def get_vectorstore():
    return Chroma(
        collection_name="mvcs_tramites_servicios",
        embedding_function=get_embeddings(),
        persist_directory=str(VECTORSTORE_DIR)
    )

def retrieve(question, k=4):
    db = get_vectorstore()
    return db.similarity_search_with_relevance_scores(question, k=k)

Overwriting src/retriever.py


In [26]:
%%writefile src/ingest.py
from pathlib import Path
from langchain_chroma import Chroma
from src.config import DOCS_DIR, VECTORSTORE_DIR
from src.loader import load_file, split_docs
from src.embeddings import get_embeddings

def index_documents():
    all_chunks = []
    for file in Path(DOCS_DIR).glob("*"):
        docs = load_file(file)
        chunks = split_docs(docs)
        for c in chunks:
            c.metadata["source_file"] = file.name
        all_chunks.extend(chunks)

    db = Chroma(
        collection_name="mvcs_tramites_servicios",
        embedding_function=get_embeddings(),
        persist_directory=str(VECTORSTORE_DIR)
    )
    db.add_documents(all_chunks)
    return len(all_chunks)

if __name__ == "__main__":
    total = index_documents()
    print(f"Chunks indexados: {total}")

Overwriting src/ingest.py


In [27]:
%%writefile src/rag_chain.py
from langchain_google_genai import ChatGoogleGenerativeAI
from src.config import GEMINI_MODEL
from src.prompts import SYSTEM_PROMPT
from src.retriever import retrieve

def ask_rag(question):
    docs = retrieve(question)
    if not docs:
        return {"answer": "No tengo evidencia suficiente en las fuentes cargadas", "sources": []}

    context = "\n\n".join([d.page_content for d, s in docs])
    sources = [{"file": d.metadata.get("source_file"), "score": s} for d, s in docs]

    llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.1)
    prompt = f"{SYSTEM_PROMPT}\n\nContexto:\n{context}\n\nPregunta:\n{question}"
    answer = llm.invoke(prompt).content

    return {"answer": answer, "sources": sources}

Overwriting src/rag_chain.py


In [28]:
!python -m src.ingest

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/rag-mvcs-streamlit/src/ingest.py", line 25, in <module>
    total = index_documents()
            ^^^^^^^^^^^^^^^^^
  File "/content/rag-mvcs-streamlit/src/ingest.py", line 21, in index_documents
    db.add_documents(all_chunks)
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores/base.py", line 258, in add_documents
    return self.add_texts(texts, metadatas, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/langchain_chroma/vectorstores.py", line 677, in add_texts
    self._collection.upsert(
  File "/usr/local/lib/python3.12/dist-packages/chromadb/api/models/Collection.py", line 521, in upsert
    upsert_request = self._validate_and_prepare_upsert_request(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/pyt

In [29]:
%%writefile app.py
import streamlit as st
from src.rag_chain import ask_rag

st.set_page_config(page_title="Asistente RAG MVCS", page_icon="🏛️", layout="wide")

st.title("🏛️ Asistente RAG MVCS")
st.caption("Consulta documentos oficiales cargados sobre trámites y servicios del MVCS.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])
        if msg.get("sources"):
            with st.expander("Fuentes utilizadas"):
                st.json(msg["sources"])

question = st.chat_input("Escribe tu pregunta sobre trámites o servicios del MVCS...")

if question:
    st.session_state.messages.append({"role": "user", "content": question})

    with st.chat_message("user"):
        st.write(question)

    with st.chat_message("assistant"):
        with st.spinner("Buscando evidencia en documentos cargados..."):
            result = ask_rag(question)

        st.write(result["answer"])

        if result.get("sources"):
            with st.expander("Fuentes utilizadas"):
                st.json(result["sources"])

    st.session_state.messages.append({
        "role": "assistant",
        "content": result["answer"],
        "sources": result.get("sources", [])
    })

Overwriting app.py


In [30]:
!ls -la app.py

-rw-r--r-- 1 root root 1303 May 15 00:44 app.py


In [31]:
%%writefile data/documentos/mvcs_procedimiento1.txt
Denominación del Procedimiento Administrativo
"APROBACIÓN DEL INFORME TÉCNICO SUSTENTADO - ITS"
Código: PA1360B3A5
---
Descripción del procedimiento
El procedimiento administrativo está a cargo de la Dirección General de Asuntos Ambientales y tiene por objeto aprobar el Informe Técnico Sustentado presentado por el administrado, el cual recoge modificatorias de su instrumento de gestión ambiental aprobado, que no generan impacto ambiental significativo.
---
Requisitos
Solicitud dirigida a la Dirección General de Asuntos Ambientales, la cual debe ser presentada antes de la implementación del proyecto.
Informe Técnico Sustentado, en el cual sustente que es necesario modificar los componentes auxiliares o hacer ampliaciones en proyectos de inversión con certificación ambiental aprobada, que tienen impacto ambiental no significativo o se pretendan hacer mejoras tecnológicas en las operaciones.
Notas:
En el marco de la Resolución Ministerial Nº 068-2021-MINAM, que aprueba la culminación del proceso de transferencia de funciones de los sectores Vivienda y Construcción del Ministerio de Vivienda, Construcción y Saneamiento – MVCS al Servicio Nacional de Certificación Ambiental para las Inversiones Sostenibles – SENACE; el MVCS atenderá sólo los procedimientos administrativos relacionados al Sector Saneamiento (ITS de la DIA y del EIA-sd) y el SENACE los Procedimientos Administrativos relacionados al Sector Vivienda y Construcciòn (ITS del EIA-d).
---
Formularios
Formulario PDF: Formulario N° 001-DGAA
Ubicación: http://sut.pcm.gob.pe/sutArchivos/file_136_20201026_163409.pdf
URL: http://nike.vivienda.gob.pe/SICA/modulos/Formulario_001.aspx
---
Canales de atención
Atención Presencial: Sede Principal y Centros de Atención al Ciudadano - CAC a Nivel Nacional
Atención Virtual: http://nike.vivienda.gob.pe/SICA/modulos/Formulario_001.aspx
---
Pago por Derecho de Tramitación
Monto - S/ 631.30
---
Modalidad de pagos
Caja de la Entidad
Efectivo: Soles (S/.), indicando el número de DNI o RUC del solicitante y el
código correspondiente (indicado por la DGAA).

Otras opciones
Agencia Bancaria: Banco de la Nación, indicando el número de DNI o RUC del
solicitante y el código correspondiente (indicado por la DGAA).
---
Plazo de atención
10 días hábiles
---
Calificación del procedimiento
Evaluación previa- Silencio Administrativo Negativo: Si vencido el plazo de atención, no obtiene respuesta, puede interponer los recursos administrativos.
---
Sedes y horarios de atención
Sede	Horario de Atención
Centro de Atención al Ciudadano - CAC de Moquegua	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Huancavelica	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Tacna	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Puno	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Loreto	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Ica	Lunes a Viernes de 08:00 a 17:00.
Sede Principal	Lunes a Viernes de 08:30 a 19:00.
Centro de Atención al Ciudadano - CAC de Arequipa	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Pasco	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Ayacucho	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Cusco	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Amazonas	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Apurímac	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Lambayeque	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Tumbes	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Cajamarca	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Ucayali	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Junín	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Madre de Dios	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de San Martín	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Piura	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Ancash	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Lima	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC de Huánuco	Lunes a Viernes de 08:00 a 17:00.
Centro de Atención al Ciudadano - CAC La Libertad	Lunes a Viernes de 08:00 a 17:00.
---
Unidad de organización donde se presenta la documentación
Oficina de Atención al Ciudadano - OAC:
Centro de Atención al Ciudadano - CAC de Moquegua
Centro de Atención al Ciudadano - CAC de Huancavelica
Centro de Atención al Ciudadano - CAC de Tacna
Centro de Atención al Ciudadano - CAC de Puno
Centro de Atención al Ciudadano - CAC de Loreto
Centro de Atención al Ciudadano - CAC de Ica
Centro de Atención al Ciudadano - CAC de Arequipa
Centro de Atención al Ciudadano - CAC de Pasco
Centro de Atención al Ciudadano - CAC de Ayacucho
Centro de Atención al Ciudadano - CAC de Cusco
Centro de Atención al Ciudadano - CAC de Amazonas
Centro de Atención al Ciudadano - CAC de Apurímac
Centro de Atención al Ciudadano - CAC de Lambayeque
Centro de Atención al Ciudadano - CAC de Tumbes
Centro de Atención al Ciudadano - CAC de Cajamarca
Centro de Atención al Ciudadano - CAC de Ucayali
Centro de Atención al Ciudadano - CAC de Junín
Centro de Atención al Ciudadano - CAC de Madre de Dios
Centro de Atención al Ciudadano - CAC de San Martín
Centro de Atención al Ciudadano - CAC de Piura
Centro de Atención al Ciudadano - CAC de Ancash
Centro de Atención al Ciudadano - CAC de Lima
Centro de Atención al Ciudadano - CAC de Huánuco
Centro de Atención al Ciudadano - CAC La Libertad
Oficina de Gestión Documentaría y Archivo - OGDA:
Sede Principal
---
Unidad de organización responsable de aprobar la solicitud
Dirección General de Asuntos Ambientales - DGAA
---
Consulta sobre el procedimiento
Teléfono: (511) 211-7930
Anexo: 6302
Correo: medio.ambiente@vivienda.gob.pe
---
Instancias de resolución de recursos
	Reconsideración	Apelación
Autoridad competente	Director(a) General - Dirección General de Asuntos Ambientales - DGAA	Viceministro(a) - Despacho Viceministerial de Construcción y Saneamiento - DVMCS
Plazo máximo de presentación	15 días hábiles	15 días hábiles
Plazo máximo de respuesta	30 días hábiles	30 días hábiles
El recurso de reconsideración se interpondrá ante el mismo órgano que dictó el primer acto que es materia de la impugnación y deberá sustentarse en nueva prueba.
El recurso de apelación se interpondrá cuando la impugnación se sustente en diferente interpretación de las pruebas producidas o cuando se trate de cuestiones de puro derecho,
debiendo dirigirse a la misma autoridad que expidió el acto que se impugna para que eleve lo actuado al superior jerárquico.
---
Base legal
Artículo	Denominación	Tipo	Número	Fecha Publicación
2, 4 y 17	Ley Marco del Sistema Nacional de Gestión Ambiental	Ley	28245	08/06/2004
52 y 58	Ley General del Ambiente	Ley	28611	15/10/2005
Párrafo 18.1 del artículo 18	Ley del Sistema Nacional de Evaluación de Impacto Ambiental	Ley	27446	23/04/2001
9	Reglamento de la Ley N° 28245, Ley Marco del Sistema Nacional de Gestión Ambiental	Decreto Supremo	008-2005-PCM	28/01/2005
Incisos a), b), e), i) del artículo 8 y Anexo II	Reglamento de la Ley N° 27446, Ley del Sistema Nacional de Evaluación de Impacto Ambiental	Decreto Supremo	019-2009-MINAM	25/09/2009
Incisos 2, 4 y 5 del artículo 5	Reglamento de Protección Ambiental para proyectos vinculados a las actividades de Vivienda, Urbanismo, Construcción y Saneamiento	Decreto Supremo	015-2012-VIVIENDA	14/09/2012
4	Disposiciones Especiales para Ejecución de Procedimientos Administrativos	Decreto Supremo	054-2013-PCM	16/05/2013
---
Fuente
TUPA: Páginas 9, 10, 11

Writing data/documentos/mvcs_procedimiento1.txt


In [32]:
!python -m src.ingest

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_google_genai/embeddings.py", line 439, in embed_documents
    result = self.client.models.embed_content(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5725, in embed_content
    return self._embed_content(model=model, contents=contents, config=config)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 4725, in _embed_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1402, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client

In [79]:
import os

# Cambiamos a gemini-2.5-flash debido a límites de cuota en 2.0
api_key = os.environ.get('GOOGLE_API_KEY')

with open('.env', 'w') as f:
    f.write(f'GOOGLE_API_KEY={api_key}\n')
    f.write('GEMINI_MODEL=models/gemini-2.5-flash\n')
    f.write('EMBEDDING_MODEL=models/gemini-embedding-001\n')

print('Archivo .env actualizado con gemini-2.5-flash.')

Archivo .env actualizado con gemini-2.5-flash.


In [58]:
# Script de Validación Final
import importlib
import src.config
import src.embeddings

# Forzamos recarga para aplicar el cambio de modelo
importlib.reload(src.config)
importlib.reload(src.embeddings)

print(f"Validando con: {src.config.EMBEDDING_MODEL}")

try:
    embed_model = src.embeddings.get_embeddings()
    test_embed = embed_model.embed_query('Hola mundo')
    print('✅ Validación exitosa: El sistema de embeddings está operativo.')
except Exception as e:
    print(f'❌ Error: {e}')

Validando con: models/gemini-embedding-001
✅ Validación exitosa: El sistema de embeddings está operativo.


In [81]:
# Re-intento de la consulta RAG con Gemini 2.5 Flash
import importlib
import src.config
import src.rag_chain
import json

# Recargamos para aplicar el cambio a gemini-2.5-flash
importlib.reload(src.config)
importlib.reload(src.rag_chain)

pregunta = "¿Qué es el Informe Técnico Sustentado (ITS) y cuánto cuesta el trámite?"
print(f"Probando RAG con modelo: {src.config.GEMINI_MODEL}\n")

try:
    resultado = src.rag_chain.ask_rag(pregunta)
    print("--- RESPUESTA ---")
    print(resultado['answer'])
    print("\n--- FUENTES ---")
    print(json.dumps(resultado['sources'], indent=2))
except Exception as e:
    print(f"❌ Error: {e}")

Probando RAG con modelo: models/gemini-2.5-flash

--- RESPUESTA ---
El Informe Técnico Sustentado (ITS) recoge modificatorias de un instrumento de gestión ambiental aprobado, que no generan impacto ambiental significativo.

El costo del trámite es de S/ 631.30.

--- FUENTES ---
[
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.6539025765691218
  },
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.5779715097148954
  },
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.5431025368580141
  },
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.49975163717886084
  }
]


In [82]:
# Paso 1: Ingesta de documentos
print("Iniciando la indexación de documentos...")
!python -m src.ingest

# Paso 2: Prueba de consulta al RAG
from src.rag_chain import ask_rag
import json

pregunta = "¿Qué es el Informe Técnico Sustentado (ITS) y cuánto cuesta el trámite?"
print(f"\nProbando RAG con la pregunta: {pregunta}\n")

resultado = ask_rag(pregunta)

print("--- RESPUESTA ---")
print(resultado['answer'])
print("\n--- FUENTES ---")
print(json.dumps(resultado['sources'], indent=2))

Iniciando la indexación de documentos...
Chunks indexados: 10

Probando RAG con la pregunta: ¿Qué es el Informe Técnico Sustentado (ITS) y cuánto cuesta el trámite?

--- RESPUESTA ---
El Informe Técnico Sustentado (ITS) es un documento presentado por el administrado que recoge modificatorias de su instrumento de gestión ambiental aprobado, las cuales no generan impacto ambiental significativo. Su objetivo es sustentar la necesidad de modificar componentes auxiliares, realizar ampliaciones en proyectos de inversión con certificación ambiental aprobada que tienen impacto ambiental no significativo, o implementar mejoras tecnológicas en las operaciones.

El costo del trámite es de S/ 631.30.

--- FUENTES ---
[
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.6539025765691218
  },
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.5779715097148954
  },
  {
    "file": "mvcs_procedimiento1.txt",
    "score": 0.5431025368580141
  },
  {
    "file": "mvcs_procedimiento1.txt",


In [67]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

print("--- Modelos de Generación (Chat/LLM) disponibles ---")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
except Exception as e:
    print(f"Error al listar modelos: {e}")

--- Modelos de Generación (Chat/LLM) disponibles ---
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/gemini-3.1-flash-tts-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-robotics-er-1.6-preview
- 

In [83]:
from src.rag_chain import ask_rag

respuesta = ask_rag("¿Sobre qué temas brinda información el MVCS?")
respuesta

{'answer': 'El MVCS atiende procedimientos administrativos relacionados al Sector Saneamiento (ITS de la DIA y del EIA-sd).',
 'sources': [{'file': 'mvcs_procedimiento1.txt', 'score': 0.5552310090134485},
  {'file': 'mvcs_procedimiento1.txt', 'score': 0.4657479812687566},
  {'file': 'mvcs_procedimiento1.txt', 'score': 0.4590377815173372},
  {'file': 'mvcs_procedimiento1.txt', 'score': 0.4552433428927499}]}

In [85]:
!pwd
!git status

/content/rag-mvcs-streamlit
On branch feature/rag-mvcs-mvp
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   app.py
	modified:   requirements.txt
	modified:   src/config.py
	modified:   src/embeddings.py
	modified:   src/ingest.py
	modified:   src/loader.py
	modified:   src/logging_config.py
	modified:   src/prompts.py
	modified:   src/rag_chain.py
	modified:   src/retriever.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/documentos/mvcs_procedimiento1.txt

no changes added to commit (use "git add" and/or "git commit -a")
